# Symbolic MIDI Generation Draft Workbook

This draft documents the current Assignment 2 pipeline for two symbolic music generation tasks: unconditioned MIDI generation and prefix-conditioned MIDI continuation. It is a working report draft, not the final exported submission.

## 1. Introduction and Task Definitions

The project treats symbolic MIDI generation as next-token language modeling over MIDI-derived event tokens. A shared model can support both required tasks:

- **Task 1: symbolic unconditioned generation.** Sample a new token sequence from a beginning seed and decode it to MIDI.
- **Task 2: symbolic prefix-conditioned continuation.** Encode a real MIDI prefix, use it as the prompt, and sample a continuation.

The main neural model is a GPT-2-style causal Transformer initialized from scratch with a custom MIDI vocabulary. No pretrained GPT-2 weights, pretrained music checkpoints, or GPT-2 text tokenizer are used.

## 2. Dataset and Preprocessing

The current draft uses the final-scale Nottingham MIDI run as the safe fallback/main route and includes a MAESTRO MIDI-only full official-split bounded experiment as a serious comparison. Audio was not used. Files are split into train/validation partitions, tokenized, and converted into fixed-length next-token windows.

### Dataset Summary

| dataset | file_count | train_files | valid_files | token_min | token_max | token_mean | train_windows | valid_windows | vocab_size | input_dir | manifest_source | skipped_file_count |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| maestro_final | 120 | 96 | 24 | 429 | 5867 | 3421.6583333333333 | 2374 | 769 | 512 | nan | nan | nan |
| maestro_full | 1099 | 962 | 137 | 416 | 53651 | 12621.800727934486 | 48133 | 5497 | 512 | data\raw\maestro_full\midi | data\raw\maestro_full\manifest.csv | 177.0 |
| nottingham_final | 3089 | 2780 | 309 | 13 | 3465 | 222.21301392036256 | 3450 | 377 | 512 | data\nottingham-dataset-master\MIDI | nan | 0.0 |

## 3. Tokenization

The primary representation is MidiTok REMI. REMI represents symbolic music with discrete musical events such as bar, position, pitch, velocity, and duration. This keeps the model in a language-modeling setting while still preserving musical timing structure.

A simple custom tokenizer remains the fallback for smoke tests if MidiTok decoding becomes unstable, but the current real-data runs use REMI successfully.

## 4. Markov / N-Gram Baseline

The Markov baseline estimates next-token probabilities from local token histories. It gives a simple, reliable reference point for valid MIDI generation and validation perplexity.

### Model Metrics

| dataset | markov_valid_perplexity | transformer_train_loss_last | transformer_valid_loss | transformer_valid_perplexity | transformer_params | block_size | steps_completed | batch_size | n_embd | n_layer | n_head | total_steps_including_resume | best_checkpoint |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| maestro_final | 281.79881948528003 | 4.671778678894043 | 4.744802231691321 | 114.98506285289204 | 478720 | 128 | 298 | nan | nan | nan | nan | nan | nan |
| maestro_full | 104.3765918074786 | 3.7378041744232178 | 3.9950605436813 | 54.32912980976267 | 3356160 | 256 | 3000 | 16.0 | 256.0 | 4.0 | 4.0 | 3512.0 | outputs\checkpoints\maestro_full\best_transformer.pt |
| nottingham_final | 16.625728607993594 | nan | 1.2181642204523089 | 3.380975307722666 | 3356160 | 256 | 0 | nan | 256.0 | 4.0 | 4.0 | 5000.0 | outputs\checkpoints\nottingham_final\best_transformer.pt |

## 5. GPT2-Style Causal Transformer Trained From Scratch

The neural model uses `GPT2Config` and `GPT2LMHeadModel(config)` as a decoder-only Transformer architecture. The model is randomly initialized and trained on MIDI token windows. The current Nottingham final checkpoint uses the same 3.36M-parameter configuration as the MAESTRO full run. The MAESTRO full bounded run used official train/validation splits, `block_size=256`, `n_embd=256`, `n_layer=4`, `n_head=4`, dropout `0.1`, and 3,512 total training steps.

## 6. MAESTRO MIDI-Only Full Experiment

MAESTRO was promoted from optional smoke test to an official-split MIDI-only comparison. The local archive contained 1,276 MIDI files. The run used 962 official train files and 137 official validation files; 177 official test files were excluded from training/evaluation and recorded as skipped. The bounded/resumed Transformer reached validation loss 3.9951 and validation perplexity 54.3291. This is a real comparison artifact, but still early-training rather than a final quality pass.

## 7. Reproducible Training Commands

The current training workflow supports full or bounded dataset runs, best-checkpoint saving, resume from checkpoint, checkpoint-only candidate generation, and candidate ranking. Long training should be run manually from PowerShell using the commands in `docs/training_commands.md`; this notebook is a draft report and does not launch long jobs itself.

## 8. Task 1: Symbolic Unconditioned Generation

For unconditioned generation, the sampler starts from a short seed and generates new MIDI tokens. Candidate files are decoded, parsed, and ranked by validity, note count, duration, pitch range, polyphony, and repetition heuristics.

## 9. Task 2: Prefix-Conditioned Continuation

For conditioned continuation, a validation MIDI prefix is used as the prompt. The model samples additional tokens after that prefix, and the resulting sequence is decoded to MIDI. This tests whether the same next-token model can generate in context.

## 10. Evaluation

Candidate MIDI files are checked for parseability and nonzero notes. The ranking table below is a rough quantitative screen, not a substitute for listening.

### Candidate Ranking

| path | dataset | task_type | model_type | temperature | top_k | candidate_index | valid | note_count | duration_seconds | notes_per_second | pitch_min | pitch_max | pitch_range | unique_pitch_count | max_simultaneous_notes | repeated_pitch_bigram_rate | score |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\nottingham_final\markov_conditioned.mid | nottingham_final | conditioned | markov | nan | nan | nan | True | 257 | 88.25 | 2.912181303116147 | 56 | 83 | 27 | 22 | 3 | 0.6328125 | 1.3281033600881336 |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\nottingham_final\markov_unconditioned.mid | nottingham_final | unconditioned | markov | nan | nan | nan | True | 270 | 88.0 | 3.0681818181818183 | 60 | 83 | 23 | 21 | 5 | 0.6394052044609665 | 1.177628985017461 |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\nottingham_final\transformer_conditioned_temp0p7_topk20_idx00.mid | nottingham_final | conditioned | transformer | 0.7 | 20.0 | 0.0 | True | 103 | 42.0 | 2.452380952380953 | 64 | 81 | 17 | 11 | 5 | 0.6764705882352942 | 0.1618405695611577 |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\nottingham_final\transformer_conditioned_temp0p7_topk20_idx01.mid | nottingham_final | conditioned | transformer | 0.7 | 20.0 | 1.0 | True | 297 | 54.5 | 5.4495412844036695 | 64 | 79 | 15 | 11 | 181 | 0.8817567567567568 | -43.50872248395184 |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\nottingham_final\transformer_conditioned_temp0p7_topk20_idx02.mid | nottingham_final | conditioned | transformer | 0.7 | 20.0 | 2.0 | True | 290 | 45.75 | 6.33879781420765 | 64 | 81 | 17 | 12 | 188 | 0.8823529411764706 | -45.43496999892854 |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\nottingham_final\transformer_conditioned_temp0p7_topk50_idx00.mid | nottingham_final | conditioned | transformer | 0.7 | 50.0 | 0.0 | True | 127 | 69.25 | 1.8339350180505416 | 64 | 79 | 15 | 11 | 3 | 0.6746031746031746 | 0.6057450146123431 |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\nottingham_final\transformer_conditioned_temp0p7_topk50_idx01.mid | nottingham_final | conditioned | transformer | 0.7 | 50.0 | 1.0 | True | 295 | 47.75 | 6.178010471204188 | 64 | 81 | 17 | 12 | 191 | 0.891156462585034 | -46.15623671807292 |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\nottingham_final\transformer_conditioned_temp0p7_topk50_idx02.mid | nottingham_final | conditioned | transformer | 0.7 | 50.0 | 2.0 | True | 295 | 45.75 | 6.448087431693989 | 64 | 79 | 15 | 11 | 191 | 0.8877551020408163 | -46.267723318835735 |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\nottingham_final\transformer_conditioned_temp0p8_topk20_idx00.mid | nottingham_final | conditioned | transformer | 0.8 | 20.0 | 0.0 | True | 115 | 42.0 | 2.738095238095238 | 64 | 81 | 17 | 12 | 8 | 0.631578947368421 | 0.2837667084377613 |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\nottingham_final\transformer_conditioned_temp0p8_topk20_idx01.mid | nottingham_final | conditioned | transformer | 0.8 | 20.0 | 1.0 | True | 111 | 59.75 | 1.8577405857740583 | 64 | 81 | 17 | 11 | 2 | 0.6454545454545455 | 0.517613108068129 |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\nottingham_final\transformer_conditioned_temp0p8_topk20_idx02.mid | nottingham_final | conditioned | transformer | 0.8 | 20.0 | 2.0 | True | 304 | 46.0 | 6.608695652173913 | 64 | 79 | 15 | 10 | 197 | 0.8877887788778878 | -47.78910412780408 |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\nottingham_final\transformer_conditioned_temp0p8_topk50_idx00.mid | nottingham_final | conditioned | transformer | 0.8 | 50.0 | 0.0 | True | 98 | 51.75 | 1.893719806763285 | 64 | 79 | 15 | 10 | 1 | 0.6804123711340206 | 0.2273998954131182 |

### MAESTRO Full Indexed Candidate Ranking

| path | dataset | task_type | model_type | temperature | top_k | candidate_index | valid | note_count | duration_seconds | notes_per_second | pitch_min | pitch_max | pitch_range | unique_pitch_count | max_simultaneous_notes | repeated_pitch_bigram_rate | score | source_path |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\maestro_full\transformer_conditioned_temp0p7_topk20_idx03.mid | maestro_full | conditioned | transformer | 0.7 | 20 | 3 | True | 72 | 23.625 | 3.0476190476190474 | 40.0 | 92.0 | 52 | 36 | 5 | 0.2112676056338028 | 1.1328219315895374 | outputs\candidates\maestro_full\transformer_conditioned_temp0p7_topk20_idx03.mid |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\maestro_full\transformer_conditioned_temp0p7_topk20_idx02.mid | maestro_full | conditioned | transformer | 0.7 | 20 | 2 | True | 67 | 4.875 | 13.743589743589745 | 40.0 | 92.0 | 52 | 36 | 5 | 0.1969696969696969 | -0.6825611888111885 | outputs\candidates\maestro_full\transformer_conditioned_temp0p7_topk20_idx02.mid |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\maestro_full\transformer_conditioned_temp0p7_topk20_idx01.mid | maestro_full | conditioned | transformer | 0.7 | 20 | 1 | True | 137 | 10.5625 | 12.970414201183432 | 40.0 | 99.0 | 59 | 38 | 12 | 0.375 | -1.4219510190664038 | outputs\candidates\maestro_full\transformer_conditioned_temp0p7_topk20_idx01.mid |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\maestro_full\transformer_conditioned_temp0p7_topk20_idx04.mid | maestro_full | conditioned | transformer | 0.7 | 20 | 4 | True | 90 | 10.375 | 8.674698795180722 | 31.0 | 97.0 | 66 | 43 | 15 | 0.2247191011235955 | -1.5228652437465215 | outputs\candidates\maestro_full\transformer_conditioned_temp0p7_topk20_idx04.mid |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\maestro_full\transformer_conditioned_temp0p7_topk20_idx00.mid | maestro_full | conditioned | transformer | 0.7 | 20 | 0 | True | 335 | 10.0 | 33.5 | 40.0 | 92.0 | 52 | 36 | 136 | 0.7574850299401198 | -36.12885894876913 | outputs\candidates\maestro_full\transformer_conditioned_temp0p7_topk20_idx00.mid |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\maestro_full\transformer_unconditioned_temp0p7_topk20_idx04.mid | maestro_full | unconditioned | transformer | 0.7 | 20 | 4 | True | 26 | 29.5 | 0.8813559322033898 | 53.0 | 53.0 | 0 | 1 | 4 | 0.96 | -1.5975188323917138 | outputs\candidates\maestro_full\transformer_unconditioned_temp0p7_topk20_idx04.mid |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\maestro_full\transformer_unconditioned_temp0p7_topk20_idx03.mid | maestro_full | unconditioned | transformer | 0.7 | 20 | 3 | True | 1 | 0.0625 | 16.0 | 62.0 | 62.0 | 0 | 1 | 1 | 0.0 | -2.0930555555555554 | outputs\candidates\maestro_full\transformer_unconditioned_temp0p7_topk20_idx03.mid |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\maestro_full\transformer_unconditioned_temp0p7_topk20_idx00.mid | maestro_full | unconditioned | transformer | 0.7 | 20 | 0 | False | 0 | 0.0 | 0.0 | nan | nan | 0 | 0 | 0 | 0.0 | -1000000000.0 | outputs\candidates\maestro_full\transformer_unconditioned_temp0p7_topk20_idx00.mid |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\maestro_full\transformer_unconditioned_temp0p7_topk20_idx01.mid | maestro_full | unconditioned | transformer | 0.7 | 20 | 1 | False | 0 | 0.0 | 0.0 | nan | nan | 0 | 0 | 0 | 0.0 | -1000000000.0 | outputs\candidates\maestro_full\transformer_unconditioned_temp0p7_topk20_idx01.mid |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\maestro_full\transformer_unconditioned_temp0p7_topk20_idx02.mid | maestro_full | unconditioned | transformer | 0.7 | 20 | 2 | False | 0 | 0.0 | 0.0 | nan | nan | 0 | 0 | 0 | 0.0 | -1000000000.0 | outputs\candidates\maestro_full\transformer_unconditioned_temp0p7_topk20_idx02.mid |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\maestro_full\transformer_conditioned_temp0p8_topk50_idx00.mid | maestro_full | conditioned | transformer | 0.8 | 50 | 0 | True | 113 | 16.625 | 6.796992481203008 | 30.0 | 92.0 | 62 | 45 | 6 | 0.1785714285714285 | 0.8142804928989138 | outputs\candidates\maestro_full\transformer_conditioned_temp0p8_topk50_idx00.mid |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\maestro_full\transformer_conditioned_temp0p8_topk50_idx01.mid | maestro_full | conditioned | transformer | 0.8 | 50 | 1 | True | 89 | 14.9375 | 5.958158995815899 | 40.0 | 92.0 | 52 | 37 | 5 | 0.1818181818181818 | 0.7648620092134736 | outputs\candidates\maestro_full\transformer_conditioned_temp0p8_topk50_idx01.mid |

### Selected Current Candidates

- nottingham_final unconditioned: `outputs\candidates\selected\nottingham_final\unconditioned_transformer.mid` (source `outputs\candidates\nottingham_final\transformer_unconditioned_temp1p0_topk20_idx00.mid`)
- nottingham_final conditioned: `outputs\candidates\selected\nottingham_final\conditioned_transformer.mid` (source `outputs\candidates\nottingham_final\transformer_conditioned_temp0p8_topk50_idx01.mid`)
- maestro_final unconditioned: `outputs\candidates\selected\maestro_final\unconditioned_transformer.mid` (source `outputs\candidates\maestro_final\transformer_unconditioned_temp1p0_topk50_idx00.mid`)
- maestro_final conditioned: `outputs\candidates\selected\maestro_final\conditioned_transformer.mid` (source `outputs\candidates\maestro_final\transformer_conditioned_temp0p8_topk20_idx01.mid`)

### MAESTRO Full Indexed Selected Candidates

- maestro_full run_001 conditioned: `outputs\candidates\final\maestro\run_001\symbolic_conditioned.mid` (source `C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\final\maestro\run_001\symbolic_conditioned.mid`)
- maestro_full run_001 unconditioned: `outputs\candidates\final\maestro\run_001\symbolic_unconditioned.mid` (source `C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\final\maestro\run_001\symbolic_unconditioned.mid`)
- maestro_full run_002 conditioned: `outputs\candidates\final\maestro\run_002\symbolic_conditioned.mid` (source `C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\final\maestro\run_002\symbolic_conditioned.mid`)
- maestro_full run_002 unconditioned: `outputs\candidates\final\maestro\run_002\symbolic_unconditioned.mid` (source `C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\final\maestro\run_002\symbolic_unconditioned.mid`)
- maestro_full run_003 conditioned: `outputs\candidates\final\maestro\run_003\symbolic_conditioned.mid` (source `C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\final\maestro\run_003\symbolic_conditioned.mid`)
- maestro_full run_003 unconditioned: `outputs\candidates\final\maestro\run_003\symbolic_unconditioned.mid` (source `C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\final\maestro\run_003\symbolic_unconditioned.mid`)

### Nottingham Final vs MAESTRO Full Selected Candidates

| path | dataset | task_type | model_type | temperature | top_k | candidate_index | valid | note_count | duration_seconds | notes_per_second | pitch_min | pitch_max | pitch_range | unique_pitch_count | max_simultaneous_notes | repeated_pitch_bigram_rate | score | run | selected_path |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\final\maestro\run_001\symbolic_conditioned.mid | maestro_full | conditioned | transformer | nan | nan | nan | True | 72 | 23.625 | 3.0476190476190474 | 40 | 92 | 52 | 36 | 5 | 0.2112676056338028 | 1.1328219315895374 | run_001 | outputs\candidates\final\maestro\run_001\symbolic_conditioned.mid |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\final\maestro\run_001\symbolic_unconditioned.mid | maestro_full | unconditioned | transformer | nan | nan | nan | True | 26 | 29.5 | 0.8813559322033898 | 53 | 53 | 0 | 1 | 4 | 0.96 | -1.5975188323917138 | run_001 | outputs\candidates\final\maestro\run_001\symbolic_unconditioned.mid |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\final\maestro\run_002\symbolic_conditioned.mid | maestro_full | conditioned | transformer | nan | nan | nan | True | 113 | 16.625 | 6.796992481203008 | 30 | 92 | 62 | 45 | 6 | 0.1785714285714285 | 0.8142804928989138 | run_002 | outputs\candidates\final\maestro\run_002\symbolic_conditioned.mid |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\final\maestro\run_002\symbolic_unconditioned.mid | maestro_full | unconditioned | transformer | nan | nan | nan | True | 5 | 8.8125 | 0.5673758865248227 | 62 | 84 | 22 | 5 | 2 | 0.0 | 0.5253841607565012 | run_002 | outputs\candidates\final\maestro\run_002\symbolic_unconditioned.mid |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\final\maestro\run_003\symbolic_conditioned.mid | maestro_full | conditioned | transformer | nan | nan | nan | True | 139 | 23.3125 | 5.962466487935657 | 40 | 92 | 52 | 37 | 9 | 0.2318840579710145 | 0.8196396886454 | run_003 | outputs\candidates\final\maestro\run_003\symbolic_conditioned.mid |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\final\maestro\run_003\symbolic_unconditioned.mid | maestro_full | unconditioned | transformer | nan | nan | nan | True | 8 | 29.625 | 0.270042194092827 | 55 | 74 | 19 | 5 | 2 | 0.1428571428571428 | 0.3617364878440828 | run_003 | outputs\candidates\final\maestro\run_003\symbolic_unconditioned.mid |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\selected\nottingham_final\conditioned_transformer.mid | nottingham_final | conditioned | transformer | nan | nan | nan | True | 139 | 149.75 | 0.9282136894824708 | 62 | 79 | 17 | 11 | 5 | 0.6376811594202898 | 0.9048419568040132 | selected | outputs\candidates\selected\nottingham_final\conditioned_transformer.mid |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\selected\nottingham_final\unconditioned_transformer.mid | nottingham_final | unconditioned | transformer | nan | nan | nan | True | 37 | 37.5 | 0.9866666666666668 | 39 | 79 | 40 | 12 | 6 | 0.25 | 0.9959166666666668 | selected | outputs\candidates\selected\nottingham_final\unconditioned_transformer.mid |

Current interpretation: Nottingham remains the safer final fallback because its validation perplexity is much lower and its selected outputs are longer and less sparse. MAESTRO full should be resumed for more steps before replacing Nottingham as the final route.

### Token Length Distributions

![Nottingham final token length distribution](../outputs/evaluation/figures/nottingham_final_token_lengths.png)

_Nottingham final token length distribution_

![MAESTRO final token length distribution](../outputs/evaluation/figures/maestro_final_token_lengths.png)

_MAESTRO final token length distribution_

![MAESTRO full token length distribution](../outputs/evaluation/maestro_full/figures/maestro_full_token_lengths.png)

_MAESTRO full token length distribution_

### Pitch-Class Histograms

![Nottingham final train vs selected generated pitch-class histogram](../outputs/evaluation/figures/nottingham_final_pitch_class_histogram.png)

_Nottingham final train vs selected generated pitch-class histogram_

![MAESTRO final train vs selected generated pitch-class histogram](../outputs/evaluation/figures/maestro_final_pitch_class_histogram.png)

_MAESTRO final train vs selected generated pitch-class histogram_

![MAESTRO full train vs selected generated pitch-class histogram](../outputs/evaluation/maestro_full/figures/maestro_full_pitch_class_histogram.png)

_MAESTRO full train vs selected generated pitch-class histogram_

## 11. Related Work Notes

This project is aligned with symbolic music generation methods from the course material, especially next-event prediction over symbolic music representations. The most relevant references for the final writeup are REMI / Pop Music Transformer, Music Transformer, Performance RNN-style symbolic sequence modeling, Markov and n-gram baselines, Nottingham, and MAESTRO.

## 12. Discussion, Limitations, and Future Work

The pipeline now produces valid MIDI candidates for both tasks. The main limitations are musical quality, heuristic candidate selection, and the fact that validation perplexity does not directly measure whether a melody is aesthetically satisfying. The next pass should listen to the selected files and add qualitative observations.

## 13. Current Artifacts and Remaining Submission Steps

Current generated artifacts live under `outputs/`, including metrics tables, figures, and selected candidate MIDI files. These are draft artifacts only. Final submission files have not been created yet.

Before submission, export this workbook to HTML, copy the selected MIDI files into `submission/` with the required names, and add the video URL file after recording the presentation.